# 04 — DQN Variants

DQN was quickly improved by three targeted fixes, each addressing a specific identified failure mode. Double DQN, Dueling DQN, and Prioritised Experience Replay (PER) each add a small amount of complexity in exchange for meaningful performance gains.

## Double DQN — fixing overestimation bias

**The problem**: vanilla DQN uses `max_{a'} Q̂(s', a'; θ⁻)` as the target. Taking the max of noisy estimates introduces an upward bias — the agent systematically overestimates Q-values, leading to suboptimal policies.

**The fix** (van Hasselt et al., 2015): decouple action *selection* from action *evaluation*:
- Use the online network θ to select the best next action: `a* = argmax_{a'} Q(s', a'; θ)`
- Use the target network θ⁻ to evaluate it: `y = r + γ Q̂(s', a*; θ⁻)`

This simple two-line change reliably reduces overestimation. The implementation difference from vanilla DQN is minimal.

## Dueling DQN — separating value and advantage

**The insight** (Wang et al., 2015): for many states, the value of being in that state is the dominant signal — the choice of action barely matters. For example, in a racing game, seeing a clear track ahead has high value regardless of which way you steer.

**The fix**: split the Q-network into two heads:
$$Q(s,a;θ) = V(s;θ_V) + \left(A(s,a;θ_A) - \frac{1}{|A|}\sum_{a'} A(s,a';θ_A)\right)$$

Where V(s) is the state value and A(s,a) is the advantage. The subtraction of the mean advantage ensures identifiability (otherwise V and A can shift freely).

Dueling helps most in environments with many actions where only a few matter in most states.

## Prioritised Experience Replay (PER)

**The problem**: uniform random sampling from the replay buffer treats all transitions equally. But high-TD-error transitions — surprising events — carry more learning signal. Uniformly sampling wastes capacity on already-learned transitions.

**The fix** (Schaul et al., 2015): sample transitions with probability proportional to their TD error |δ|:
$$P(i) \propto |\delta_i|^\alpha$$

α=0 is uniform; α=1 is full prioritisation. To correct for the sampling bias introduced by non-uniform sampling, importance sampling weights `w_i = (N·P(i))^{-β}` are applied, with β annealed from a small value to 1 over training.

In practice: PER + Double DQN + Dueling = **Rainbow DQN** (Hessel et al., 2017), which held state-of-the-art on Atari for some time. Most of the gain comes from distributional RL (not covered here) and PER; Dueling and Double DQN are smaller contributions.

In [ ]:
# Double DQN: the two-line fix
# Compare vanilla DQN vs Double DQN on CartPole

import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import numpy as np, random
from collections import deque
import gymnasium as gym
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42); random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class QNet(nn.Module):
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(s_dim,128),nn.ReLU(),nn.Linear(128,128),nn.ReLU(),nn.Linear(128,a_dim))
    def forward(self, x): return self.net(x)

class DuelingQNet(nn.Module):
    """Value + Advantage streams with shared feature extractor.""""
    def __init__(self, s_dim, a_dim):
        super().__init__()
        self.feature = nn.Sequential(nn.Linear(s_dim,128),nn.ReLU())
        self.value = nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,1))
        self.advantage = nn.Sequential(nn.Linear(128,64),nn.ReLU(),nn.Linear(64,a_dim))

    def forward(self, x):
        feat = self.feature(x)
        v = self.value(feat)
        a = self.advantage(feat)
        return v + (a - a.mean(dim=1, keepdim=True))  # mean advantage subtraction

def train_dqn(use_double=False, use_dueling=False, n_episodes=400, seed=42):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    env = gym.make("CartPole-v1")
    s_dim, a_dim = env.observation_space.shape[0], env.action_space.n

    NetClass = DuelingQNet if use_dueling else QNet
    q_net = NetClass(s_dim, a_dim).to(device)
    target_net = NetClass(s_dim, a_dim).to(device)
    target_net.load_state_dict(q_net.state_dict()); target_net.eval()
    optimizer = optim.Adam(q_net.parameters(), lr=1e-3)
    buffer = deque(maxlen=10_000)

    epsilon = 1.0
    returns = []

    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed+ep)
        total_r = 0
        for _ in range(500):
            if random.random() < epsilon:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    action = q_net(torch.FloatTensor(state).to(device)).argmax().item()

            ns, r, term, trunc, _ = env.step(action)
            done = term or trunc
            buffer.append((state, action, r, ns, float(done)))
            state = ns; total_r += r
            if done: break

            if len(buffer) >= 64:
                batch = random.sample(buffer, 64)
                s,a,rw,ns_,dn = [torch.FloatTensor(np.array(x)).to(device) for x in zip(*batch)]
                a = a.long()

                q_vals = q_net(s).gather(1, a.unsqueeze(1)).squeeze(1)

                with torch.no_grad():
                    if use_double:
                        # Double DQN: online net selects action, target net evaluates
                        next_actions = q_net(ns_).argmax(1)
                        next_q = target_net(ns_).gather(1, next_actions.unsqueeze(1)).squeeze(1)
                    else:
                        # Vanilla DQN: target net both selects and evaluates
                        next_q = target_net(ns_).max(1)[0]
                    targets = rw + 0.99 * next_q * (1 - dn)

                loss = F.mse_loss(q_vals, targets)
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(q_net.parameters(), 10)
                optimizer.step()

        epsilon = max(0.01, epsilon * 0.995)
        if ep % 100 == 0:
            target_net.load_state_dict(q_net.state_dict())
        returns.append(total_r)

    env.close()
    return returns

print("Training variants... (this takes ~2 minutes)")
r_vanilla = train_dqn(use_double=False, use_dueling=False)
r_double  = train_dqn(use_double=True,  use_dueling=False)
r_dueling = train_dqn(use_double=False, use_dueling=True)
r_both    = train_dqn(use_double=True,  use_dueling=True)

def smooth(x, w=20): return np.convolve(x, np.ones(w)/w, mode='valid')

plt.figure(figsize=(10, 5))
for returns, label, color in [
    (r_vanilla, 'Vanilla DQN', 'gray'),
    (r_double,  'Double DQN', 'steelblue'),
    (r_dueling, 'Dueling DQN', 'darkorange'),
    (r_both,    'Double + Dueling', 'green'),
]:
    plt.plot(smooth(returns), label=label, color=color)
plt.axhline(195, color='red', linestyle='--', alpha=0.5, label='Solved (195)')
plt.xlabel('Episode'); plt.ylabel('Return (smoothed)')
plt.title('DQN Ablation: Vanilla vs Double vs Dueling vs Both', fontweight='bold')
plt.legend(); plt.tight_layout()
plt.savefig('/tmp/dqn_ablation.png', dpi=100, bbox_inches='tight')
plt.show()

final_avgs = {
    'Vanilla': np.mean(r_vanilla[-50:]),
    'Double': np.mean(r_double[-50:]),
    'Dueling': np.mean(r_dueling[-50:]),
    'Double+Dueling': np.mean(r_both[-50:]),
}
print("\nFinal 50-episode averages:")
for k, v in final_avgs.items():
    print(f"  {k:<20}: {v:.1f}")


## When to use DQN variants in practice

For new projects, DQN variants are rarely the first choice. PPO or SAC handle most tasks better with less tuning. DQN variants remain relevant for:
- **Discrete action spaces** where you explicitly want Q-value estimates (e.g., game-playing, bandit-like settings)
- **Off-policy learning from fixed datasets** (offline RL) — replay buffer + target network is the right inductive structure
- **Combinatorial action spaces** with structured decomposition

If you need Q-learning specifically, start with Double + Dueling as defaults. Add PER if sample efficiency matters and you have time to tune the priority exponent.